<h1>Librerias</h1>

In [ ]:
from sklearn import datasets
import warnings
warnings.filterwarnings('ignore')

# Para operaciones básicas de manipulación de datos
import numpy as np 
import pandas as pd 

# Para visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns
from pandas import plotting
from pandas.plotting import parallel_coordinates

#Normalice las columnas con el siguiente código:
from sklearn.preprocessing import MinMaxScaler

# para modelación
import sklearn
#import imblearn

from sklearn.metrics import r2_score , mean_absolute_error, mean_squared_error,accuracy_score,classification_report,confusion_matrix

from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from viznet import connecta2a, node_sequence, NodeBrush, EdgeBrush, DynamicShow


<h1>Datos</h1>
Realizaremos un clasificador de vinos basandonos en 13 características de estos. Las categorias son 0, 1 y 2.<br/>
El conjunto de datos sobre vinos contiene los resultados de un análisis químico de vinos cultivados en una zona específica de Italia. En las 178 muestras están representados tres tipos de vino, y para cada muestra se registran los resultados de 13 análisis químicos. La variable Tipo se ha transformado en una variable categórica.<br/>
Los datos no contienen valores faltantes y consisten únicamente en datos numéricos, con una variable objetivo de tres clases (Tipo) para la clasificación.

In [ ]:
wine=datasets.load_wine(as_frame=True)
wine.keys()

In [ ]:
df=wine.frame
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

<h1>Análisis de correlación</h1>

In [ ]:
print("Matriz de Correlación")
plt.rcParams['figure.figsize']=(15,6)
sns.heatmap(df.corr(),cmap='coolwarm',linewidths=.5,fmt=".2f",annot = True);

In [ ]:
###Elimine las columnas que están altamente correlacionadas con el siguiente código:
X=df.drop(columns=['target'],axis=1)
corr = X.corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(np.bool_))
drop = [column for column in upper.columns if any(np.abs(upper[column]) > 0.7)]
print(f"columns dropped are: {drop}")
df.drop(columns=drop, axis=1, inplace=True)
X.drop(columns=drop, axis=1, inplace=True)

In [ ]:
viz_df = df.copy()
median_home_price = viz_df['target'].median().round(2)
print(median_home_price)
k = 0
plt.figure(figsize=(20, 30))
for col in viz_df.drop('target', axis=1).columns:
    plt.subplot(6, 3, k + 1)
    if viz_df[col].dtype == 'float64' or viz_df[col].dtype =='int8':
        sns.scatterplot(data=viz_df, x=col, y='target')
        plt.title(f"Tipo de Vino Vs {col}")
    else:
        group = viz_df[[col, 'target']].groupby(by=col).median()
        sns.barplot(data=group, x=group.index, y='target')
        plt.axhline(y=median_home_price, label='Tipo de Vino', color='black', linestyle='--', linewidth=0.7)
        plt.title(f"Tipo de Vino by {col}")
        plt.legend()
    k += 1
plt.tight_layout()

In [ ]:
ax = df['target'].value_counts().plot(kind='bar', figsize=(8, 10), fontsize=13, color=["red","blue","green"])
ax.set_title('Tipo de vino', size=20, pad=30)
ax.set_ylabel('Tipo de vino', fontsize=10)

for i in ax.patches:
    ax.text(i.get_x() + 0.19, i.get_height() + 1, str(round(i.get_height(), 2)), fontsize=15)

In [ ]:
a=pd.DataFrame(df['target'].value_counts())
a.reset_index(inplace=True)

a.plot(kind='pie',y='target',labels=a['index'],autopct='%1.2f')
plt.show()

In [ ]:
X = df.drop(['target'], axis = 1)
y = df['target']

<h1>Escalación</h1>

In [ ]:
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(X.to_numpy())
df_scaled = pd.DataFrame(df_scaled,columns=X.columns)

<h1>Separamos en datos de entrenamiento y datos de prueba</h1>

In [ ]:
# let's split the dataset into train and test sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 29)

<h1>Modelo de redes neuronales con SKlearn</h1>

In [ ]:
from sklearn.neural_network import MLPClassifier
nn = MLPClassifier(
    activation=('relu'),
    hidden_layer_sizes=70,
    alpha=0.001,
    random_state=20,
    early_stopping=False
)
#
# Train the model
#
nn.fit(X_train, y_train)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error
 
# Make prediction
predicciones = nn.predict(X_test)
# Accuracy de test del modelo 
# ==============================================================================
accuracy = accuracy_score(
            y_true    = y_test,
            y_pred    = predicciones,
            normalize = True
           )
print("")
print(f"El accuracy de test es: {100*accuracy}%")

In [ ]:
# making a classification report
cr = classification_report(y_test,  predicciones)
print(cr)

sns.set(font_scale=1)
# making a confusion matrix
plt.rcParams['figure.figsize'] = (5, 5)
cm = confusion_matrix(y_test, predicciones)
sns.heatmap(cm, annot = True, cmap = 'spring')
plt.show()

<H1>Optimizando los hiper parámetros</H1>

In [ ]:
# Número de neuronas
# ==============================================================================
param_grid = {'hidden_layer_sizes':[50,60,70,80,90,
                                    (5,5),(10,10),(20,20),(40,40),(60,60),
                                   (10,10,5),(20,20,10)],
              'learning_rate_init':[0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
             'activation': ['logistic', 'tanh', 'relu']}#'identity', 

grid = GridSearchCV(
        estimator = MLPClassifier(
                        solver = 'lbfgs',#para datos pequenos, en caso contrario usar adam
                        random_state = 123
                    ),
        param_grid = param_grid,
        scoring    = 'accuracy',
        cv         = 5,
        refit      = True,
        return_train_score = True
      )

_ = grid.fit(X, y)

In [ ]:
resultados = pd.DataFrame(grid.cv_results_)
resultados.filter(regex = '(param.*|mean_t|std_t)')\
    .drop(columns = 'params')\
    .sort_values('mean_test_score', ascending = False) \
    .head(60)

<h1>Probando la mejor configuración</h1>

In [ ]:
nn = MLPClassifier(
    activation=('relu'),
    hidden_layer_sizes=70,
    random_state=20,#None,#123,
    alpha=0.001,
    early_stopping=False,
    learning_rate_init=0.001,
    solver = 'lbfgs',
)

# Train the model
nn.fit(X_train, y_train)

In [ ]:
# Make prediction
predicciones = nn.predict(X_test)
# Accuracy de test del modelo 
# ==============================================================================
accuracy = accuracy_score(
            y_true    = y_test,
            y_pred    = predicciones,
            normalize = True
           )
print("")
print(f"El accuracy de test es: {100*accuracy}%")

In [ ]:
# making a classification report
cr = classification_report(y_test,  predicciones)
print(cr)

sns.set(font_scale=1)
# making a confusion matrix
plt.rcParams['figure.figsize'] = (5, 5)
cm = confusion_matrix(y_test, predicciones)
sns.heatmap(cm, annot = True, cmap = 'spring')
plt.show()

<h1>Keras con tensorflow</h1>

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import set_random_seed

model = Sequential()
set_random_seed(123)
model.add(Dense(70, activation='tanh'))#70: 89,x90 89, 50: 92, 60x, 55:94
model.add(Dense(1, activation='relu'))
 
model.compile(loss='mean_squared_error',
              optimizer='adam',
              metrics=['binary_accuracy'])
 
model.fit(X_train, y_train, epochs=600)
 
# evaluamos el modelo
scores = model.evaluate(X_test, y_test)
 
print("\n%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
#print (model.predict(X_test).round())

In [ ]:
predictions=model.predict(X_test).round()
# making a classification report
cr = classification_report(y_test,  predictions)
print(cr)

sns.set(font_scale=1)
# making a confusion matrix
plt.rcParams['figure.figsize'] = (5, 5)
cm = confusion_matrix(y_test, predictions)
sns.heatmap(cm, annot = True, cmap = 'spring')
plt.show()

<h1>Estructura de la red neuronal</h1>

In [ ]:
def draw_feed_forward(ax, num_node_list):
    '''
    draw a feed forward neural network.

    Args:
        num_node_list (list<int>): number of nodes in each layer.
    '''
    num_hidden_layer = len(num_node_list) - 2
    token_list = ['\sigma^z'] + \
        ['y^{(%s)}' % (i + 1) for i in range(num_hidden_layer)] + ['\psi']
    kind_list = ['nn.input'] + ['nn.hidden'] * num_hidden_layer + ['nn.output']
    radius_list = [0.3] + [0.1] * num_hidden_layer + [0.3]
    y_list = 10.5 * np.arange(len(num_node_list))

    seq_list = []
    for n, kind, radius, y in zip(num_node_list, kind_list, radius_list, y_list):
        b = NodeBrush(kind, ax)
        seq_list.append(node_sequence(b, n, center=(0, y)))

    eb = EdgeBrush('-->', ax)
    for st, et in zip(seq_list[:-1], seq_list[1:]):
        connecta2a(st, et, eb)


def real_bp():
    with DynamicShow((10, 4), '_feed_forward.png') as d:
        draw_feed_forward(d.ax, num_node_list=[11,55, 1])#configurar red


if __name__ == '__main__':
    real_bp()

In [ ]:
weights, biases = model.layers[0].get_weights()

In [ ]:
weights

In [ ]:
weights.shape

In [ ]:
biases

In [ ]:
biases.shape

In [ ]:
weights, biases = model.layers[1].get_weights()

In [ ]:
weights

In [ ]:
weights.shape

<h1>Guardar y leer red neuronal</h1>
<a href="https://keras.io/api/saving/">https://keras.io/api/saving/</a>